# ML Testing & MLOps: Building a Validation & Monitoring Suite for a Housing Price Regressor

**Problem statement.** You've just joined an ML team as the embedded QA engineer for a model that predicts median house values from the California Housing dataset (real 1990 census data, `sklearn.datasets.fetch_california_housing`). The data scientists have a notebook that trains a `RandomForestRegressor`, but there is no automated way to catch (a) bad or malformed input data before it reaches the model, (b) a model whose accuracy has silently regressed, or (c) a newly-arrived batch of houses whose feature distributions look meaningfully different from the data the model was trained on (feature drift). Your job, exactly as it would be for a tabular-data SQA role moving into ML quality, is to turn "looks fine to me" into a small battery of automated, re-runnable checks.

**What you should produce**, as runnable functions (not just prose) in the cells below:

1. `validate_schema(df)` — a data-validation function that checks column presence, dtypes, and that every numeric column's values fall inside a documented, sane range (e.g. `MedInc >= 0`, `HouseAge >= 0`, `Latitude` within California's bounds, no nulls). It should return a list of human-readable violation strings (empty list = pass).
2. A trained baseline `RandomForestRegressor` plus `evaluate_model(model, X, y)` returning a dict of `{mae, rmse, r2}`, and `check_performance_thresholds(metrics)` that asserts the model clears a minimum bar (you choose and justify the bar from the baseline's own numbers) and returns pass/fail + reasons.
3. `detect_drift(reference_df, current_df, features, alpha=0.05)` — uses a two-sample Kolmogorov-Smirnov test (`scipy.stats.ks_2samp`) per feature to flag which features differ significantly between a "reference" cohort and a "current" cohort. You will build the two cohorts by splitting the **real** dataset geographically (coastal vs. inland counties, using `Longitude`) to simulate a model trained on one region being rolled out to houses from a new region — a realistic MLOps scenario, not injected synthetic noise.
4. `run_quality_gate(...)` — a single function that calls all of the above and prints/returns a consolidated pass/fail report, the way a CI gate for a model-deployment pipeline would.

Work through the starter cells first; the solution (with explanations) is collapsed further down so you can check your approach after attempting it yourself.


## Setup and data loading

Run this cell as-is. It loads the real California Housing dataset (20,640 rows, 1990 census block groups) and gives you a DataFrame `df` with the 8 feature columns plus the target `MedHouseVal`.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import ks_2samp

housing = fetch_california_housing(as_frame=True)
df = housing.frame.rename(columns={"MedHouseVal": "MedHouseVal"})
print(df.shape)
df.head()


## Part 1 — Data validation

Fill in `validate_schema` below. It should catch:
- missing expected columns,
- any null values,
- out-of-range values for at least: `MedInc` (>= 0), `HouseAge` (>= 0), `AveRooms` (> 0), `Population` (>= 0), `Latitude` (32.0–42.5, California's real bounds), `Longitude` (-124.5 to -114.0).

Return a **list of violation strings**. An empty list means the batch passed validation.


In [ ]:
EXPECTED_COLUMNS = ["MedInc", "HouseAge", "AveRooms", "AveBedrms", "Population",
                    "AveOccup", "Latitude", "Longitude", "MedHouseVal"]

RANGE_RULES = {
    "MedInc": (0, None),
    "HouseAge": (0, None),
    "AveRooms": (0, None),
    "Population": (0, None),
    "Latitude": (32.0, 42.5),
    "Longitude": (-124.5, -114.0),
}

def validate_schema(df: pd.DataFrame) -> list[str]:
    # TODO: return a list of violation strings; [] means the dataframe passed.
    violations = []
    # TODO: check EXPECTED_COLUMNS are all present
    # TODO: check for nulls in any column
    # TODO: for each column in RANGE_RULES, check min/max bounds are respected
    raise NotImplementedError

# Try it on the clean data -- should return []
# validate_schema(df)

# Now try it on a deliberately corrupted copy -- should return several violations
bad_df = df.copy()
bad_df.loc[0, "HouseAge"] = -5
bad_df.loc[1, "Latitude"] = 90.0
bad_df.loc[2, "MedInc"] = np.nan
# validate_schema(bad_df)


## Part 2 — Model performance gate

Train a baseline model, write `evaluate_model`, and write `check_performance_thresholds` that turns raw metrics into a pass/fail decision a CI pipeline could act on.


In [ ]:
X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

baseline_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
baseline_model.fit(X_train, y_train)

def evaluate_model(model, X, y) -> dict:
    # TODO: return {"mae": ..., "rmse": ..., "r2": ...} for model's predictions on X, y.
    raise NotImplementedError

def check_performance_thresholds(metrics: dict, max_mae: float, min_r2: float) -> dict:
    # TODO: return {"passed": bool, "reasons": [str, ...]} comparing metrics against the given thresholds.
    raise NotImplementedError

# metrics = evaluate_model(baseline_model, X_test, y_test)
# print(metrics)
# check_performance_thresholds(metrics, max_mae=..., min_r2=...)


## Part 3 — Drift detection across a real geographic split

Build two **real** cohorts from the dataset itself (no synthetic data needed): houses west of `Longitude = -121.0` ("coastal", roughly the Bay Area / coastal strip) as the **reference** cohort the model was "trained on", and houses east of that line ("inland", e.g. Central Valley / inland counties) as the **current** cohort representing a new region the model is now being served on. This is a realistic proxy for geographic/population drift when a model is rolled out beyond its original training region.

Write `detect_drift` using `scipy.stats.ks_2samp` per feature.


In [ ]:
reference_cohort = df[df["Longitude"] < -121.0].drop(columns=["MedHouseVal"])
current_cohort = df[df["Longitude"] >= -121.0].drop(columns=["MedHouseVal"])
print("reference:", reference_cohort.shape, "current:", current_cohort.shape)

def detect_drift(reference_df: pd.DataFrame, current_df: pd.DataFrame,
                  features: list[str], alpha: float = 0.05) -> pd.DataFrame:
    # TODO: for each feature, run ks_2samp(reference_df[f], current_df[f]) and
    # return a DataFrame with columns [feature, ks_statistic, p_value, drifted]
    # where drifted = p_value < alpha.
    raise NotImplementedError

# detect_drift(reference_cohort, current_cohort, list(reference_cohort.columns))


## Part 4 — The quality gate

Write `run_quality_gate` that ties Parts 1–3 together into one CI-style report.


In [ ]:
def run_quality_gate(df, model, X_test, y_test, reference_cohort, current_cohort,
                      max_mae, min_r2, drift_alpha=0.05) -> dict:
    # TODO:
    # 1. run validate_schema(df) -> schema_violations
    # 2. run evaluate_model + check_performance_thresholds -> perf_result
    # 3. run detect_drift(...) -> drift_result (list the drifted feature names)
    # 4. combine into one dict: {"schema_ok": bool, "performance_ok": bool,
    #    "drifted_features": [...], "overall_pass": bool} and print a short report
    raise NotImplementedError

# run_quality_gate(df, baseline_model, X_test, y_test, reference_cohort, current_cohort,
#                   max_mae=0.45, min_r2=0.75)


---

## Solution

*(scroll down when you're ready — try it yourself first)*


<details>
<summary>Click to reveal solution</summary>

**Part 1 — `validate_schema`**

```python
def validate_schema(df: pd.DataFrame) -> list[str]:
    violations = []

    missing = set(EXPECTED_COLUMNS) - set(df.columns)
    if missing:
        violations.append(f"Missing expected columns: {sorted(missing)}")

    null_counts = df.isnull().sum()
    for col, n in null_counts[null_counts > 0].items():
        violations.append(f"Column '{col}' has {n} null value(s)")

    for col, (lo, hi) in RANGE_RULES.items():
        if col not in df.columns:
            continue
        if lo is not None and (df[col] < lo).any():
            n = (df[col] < lo).sum()
            violations.append(f"Column '{col}' has {n} value(s) below minimum {lo}")
        if hi is not None and (df[col] > hi).any():
            n = (df[col] > hi).sum()
            violations.append(f"Column '{col}' has {n} value(s) above maximum {hi}")

    return violations
```

`validate_schema(df)` returns `[]` on the clean dataset, and on `bad_df` it reports the negative `HouseAge`, the impossible `Latitude`, and the null `MedInc` — three independent, specific violations rather than a single "something's wrong" flag. That specificity is the point: a QA check that just returns `False` forces someone to re-investigate from scratch.

**Part 2 — performance gate**

```python
def evaluate_model(model, X, y) -> dict:
    preds = model.predict(X)
    return {
        "mae": mean_absolute_error(y, preds),
        "rmse": mean_squared_error(y, preds) ** 0.5,
        "r2": r2_score(y, preds),
    }

def check_performance_thresholds(metrics: dict, max_mae: float, min_r2: float) -> dict:
    reasons = []
    if metrics["mae"] > max_mae:
        reasons.append(f"MAE {metrics['mae']:.4f} exceeds max allowed {max_mae}")
    if metrics["r2"] < min_r2:
        reasons.append(f"R2 {metrics['r2']:.4f} below minimum required {min_r2}")
    return {"passed": len(reasons) == 0, "reasons": reasons}

metrics = evaluate_model(baseline_model, X_test, y_test)
print(metrics)
print(check_performance_thresholds(metrics, max_mae=0.45, min_r2=0.75))
```

The baseline `RandomForestRegressor` on this split typically lands around MAE ≈ 0.33, RMSE ≈ 0.50, R² ≈ 0.80, so thresholds of `max_mae=0.45` / `min_r2=0.75` give some headroom above the current baseline while still catching a real regression (e.g. someone accidentally training on a shuffled target, or swapping in too few trees). **The key QA habit:** thresholds are chosen *relative to a measured baseline*, not pulled from thin air — and they should be loose enough to tolerate normal retraining noise but tight enough to catch a real break.

**Part 3 — drift detection**

```python
def detect_drift(reference_df: pd.DataFrame, current_df: pd.DataFrame,
                  features: list[str], alpha: float = 0.05) -> pd.DataFrame:
    rows = []
    for f in features:
        stat, p = ks_2samp(reference_df[f], current_df[f])
        rows.append({"feature": f, "ks_statistic": stat, "p_value": p, "drifted": p < alpha})
    return pd.DataFrame(rows).sort_values("p_value")

drift_report = detect_drift(reference_cohort, current_cohort, list(reference_cohort.columns))
print(drift_report)
```

Because the two cohorts are real coastal vs. inland California block groups, several features drift hard and for sensible reasons: `MedInc` and `Population` differ (coastal areas are wealthier and denser), and `Latitude`/`Longitude` drift by construction of the split. `AveRooms`/`AveBedrms` tend to show milder or no drift. This mirrors what you'd see rolling a model out to a new metro area: some features shift a lot, others barely move, and a quality gate should name *which* features drifted rather than emitting one boolean.

**Part 4 — quality gate**

```python
def run_quality_gate(df, model, X_test, y_test, reference_cohort, current_cohort,
                      max_mae, min_r2, drift_alpha=0.05) -> dict:
    schema_violations = validate_schema(df)

    metrics = evaluate_model(model, X_test, y_test)
    perf_result = check_performance_thresholds(metrics, max_mae, min_r2)

    drift_df = detect_drift(reference_cohort, current_cohort,
                             list(reference_cohort.columns), alpha=drift_alpha)
    drifted_features = drift_df.loc[drift_df["drifted"], "feature"].tolist()

    report = {
        "schema_ok": len(schema_violations) == 0,
        "schema_violations": schema_violations,
        "performance_ok": perf_result["passed"],
        "performance_metrics": metrics,
        "drifted_features": drifted_features,
        "overall_pass": len(schema_violations) == 0 and perf_result["passed"],
    }

    print("=== Quality Gate Report ===")
    print(f"Schema check:      {'PASS' if report['schema_ok'] else 'FAIL'}")
    print(f"Performance check: {'PASS' if report['performance_ok'] else 'FAIL'} ({metrics})")
    print(f"Drifted features:  {drifted_features or 'none'}  (informational, not a hard fail)")
    print(f"OVERALL:           {'PASS' if report['overall_pass'] else 'FAIL'}")
    return report

run_quality_gate(df, baseline_model, X_test, y_test, reference_cohort, current_cohort,
                  max_mae=0.45, min_r2=0.75)
```

Note the deliberate design choice: schema violations and performance regressions **block** the gate (`overall_pass`), while drift is reported but doesn't automatically fail the build. That's a realistic MLOps judgment call — drift is a signal to investigate and potentially retrain, not necessarily a reason to block the current deployment, since some drift (like a genuine regional rollout) is expected and acceptable. Deciding which checks are hard gates versus soft alerts is itself a core quality-engineering skill when moving from traditional SQA into ML systems.

</details>
